# Flood Mapping and Vulnerability Assessment in River and Lagoon Environments

*by Daniele Petrini & Federico Urbini*

# Datasets & Setup

## Necessary Imports

In [1]:
import ee
import geemap
import pandas as pd
from pathlib import Path
from utils import utils_setup as us

/home/federico/miniconda3/envs/floods/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


## Authentication and Folders

In [2]:
ee.Authenticate(auth_mode="localhost")
ee.Initialize(project="floods-2024")
print("Earth Engine initialized correctly.")

Earth Engine initialized correctly.


In [3]:
# Create output folders
Path("outputs").mkdir(exist_ok=True)
Path("outputs/tables").mkdir(parents=True, exist_ok=True)
Path("outputs/figures").mkdir(parents=True, exist_ok=True)
Path("outputs/maps").mkdir(parents=True, exist_ok=True)

## Flags

In [4]:
INCLUDE_SEASONAL_REFERENCE = True 
SHOW_RIVERS_BY_DEFAULT = False 
SHOW_S2_PREVIEWS_BY_DEFAULT = False
SHOW_WATER_ON_EVENT_MAP = False

AUX_DEFAULT_LAYER = "JRC"
# Options: "JRC", "PERMANENT_WATER", "WORLDCOVER", "DEM", "SLOPE", "NONE"

## Geographic Areas and Dates

In [5]:
# Define study areas, places of interest, event windows, and event metadata.

aoi_emilia_romagna = ee.Geometry.Rectangle([10.7, 43.7, 12.8, 45.1])
aoi_romagna = ee.Geometry.Rectangle([11.3, 44.0, 12.5, 44.8])
aoi_bologna = ee.Geometry.Rectangle([11.15, 44.15, 11.65, 44.65])

places = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([11.8833, 44.2856]), {"name": "Faenza"}),
    ee.Feature(ee.Geometry.Point([11.8000, 44.3190]), {"name": "Castel Bolognese"}),
    ee.Feature(ee.Geometry.Point([11.9100, 44.4214]), {"name": "Lugo"}),
    ee.Feature(ee.Geometry.Point([11.9781, 44.4142]), {"name": "Bagnacavallo"}),
    ee.Feature(ee.Geometry.Point([12.2035, 44.4184]), {"name": "Ravenna"}),
    ee.Feature(ee.Geometry.Point([11.3426, 44.4949]), {"name": "Bologna"}),
    ee.Feature(ee.Geometry.Point([11.3436, 44.3896]), {"name": "Pianoro"}),
])

dates = {
    "sept_pre": ("2024-08-20", "2024-09-15"),
    "sept_post": ("2024-09-18", "2024-09-25"),
    "sept_recovery": ("2024-09-26", "2024-10-15"),
    "oct_pre": ("2024-10-01", "2024-10-18"),
    "oct_post": ("2024-10-20", "2024-10-27"),
    "oct_recovery": ("2024-10-28", "2024-11-15"),
    "seasonal_ref": ("2023-08-20", "2023-09-30"),
}

events = {
    "sept": {
        "name": "September / Romagna",
        "aoi": aoi_romagna,
        "pre": "sept_pre",
        "post": "sept_post",
        "recovery": "sept_recovery",
        "center": [44.35, 11.9],
        "zoom": 9,
        "color": "red",
        "aoi_label": "AOI Romagna",
    },
    "oct": {
        "name": "October / Bologna-Pianoro",
        "aoi": aoi_bologna,
        "pre": "oct_pre",
        "post": "oct_post",
        "recovery": "oct_recovery",
        "center": [44.43, 11.35],
        "zoom": 10,
        "color": "blue",
        "aoi_label": "AOI Bologna/Pianoro",
    },
}

In [6]:
# Create transparent AOI outlines and styled city points.
aoi_styles = {
    "Broad AOI": ee.FeatureCollection([ee.Feature(aoi_emilia_romagna)]).style(color="yellow", fillColor="00000000", width=2),
    "AOI Romagna": ee.FeatureCollection([ee.Feature(aoi_romagna)]).style(color="red", fillColor="00000000", width=2),
    "AOI Bologna/Pianoro": ee.FeatureCollection([ee.Feature(aoi_bologna)]).style(color="blue", fillColor="00000000", width=2),
}

event_aoi_styles = {
    "sept": aoi_styles["AOI Romagna"],
    "oct": aoi_styles["AOI Bologna/Pianoro"],
}

places_styles = {
    "emilia": places.filterBounds(aoi_emilia_romagna).style(color="000000", pointSize=5,pointShape="circle"),
    "sept": places.filterBounds(aoi_romagna).style(color="000000", pointSize=5, pointShape="circle"),
    "oct": places.filterBounds(aoi_bologna).style(color="000000", pointSize=5, pointShape="circle"),
}

## Load Collections and Datasets

In [7]:
# Load Sentinel-1 GRD collections for each event and orbit direction.

s1 = {}
s1_orbits = {}

for event_key, event in events.items():
    aoi = event["aoi"]

    s1[event_key] = {
        "pre": us.load_sentinel1(aoi, *dates[event["pre"]]),
        "post": us.load_sentinel1(aoi, *dates[event["post"]]),
        "recovery": us.load_sentinel1(aoi, *dates[event["recovery"]]),
    }

    s1_orbits[event_key] = {
        "pre_ASC": us.load_sentinel1(aoi, *dates[event["pre"]], orbit_pass="ASCENDING"),
        "post_ASC": us.load_sentinel1(aoi, *dates[event["post"]], orbit_pass="ASCENDING"),
        "pre_DESC": us.load_sentinel1(aoi, *dates[event["pre"]], orbit_pass="DESCENDING"),
        "post_DESC": us.load_sentinel1(aoi, *dates[event["post"]], orbit_pass="DESCENDING"),
    }

if INCLUDE_SEASONAL_REFERENCE:
    s1_seasonal_ref = {
        "ALL": us.load_sentinel1(aoi_romagna, *dates["seasonal_ref"]),
        "ASC": us.load_sentinel1(aoi_romagna, *dates["seasonal_ref"], orbit_pass="ASCENDING"),
        "DESC": us.load_sentinel1(aoi_romagna, *dates["seasonal_ref"], orbit_pass="DESCENDING"),
    }

In [8]:
# Load raw Sentinel-2 Surface Reflectance collections for each event.

s2 = {}

for event_key, event in events.items():
    aoi = event["aoi"]

    s2[event_key] = {
        "pre": us.load_sentinel2_raw(aoi, *dates[event["pre"]], max_cloud=95),
        "post": us.load_sentinel2_raw(aoi, *dates[event["post"]], max_cloud=100),
        "recovery": us.load_sentinel2_raw(aoi, *dates[event["recovery"]], max_cloud=95),
    }

if INCLUDE_SEASONAL_REFERENCE:
    s2_seasonal_ref_romagna = us.load_sentinel2_raw(
        aoi_romagna,
        *dates["seasonal_ref"],
        max_cloud=95
    )

In [9]:
# Load permanent water, land cover, DEM, slope, and river datasets.

auxiliary = us.load_auxiliary_datasets(aoi_emilia_romagna)

water_occurrence = auxiliary["water_occurrence"]
permanent_water = auxiliary["permanent_water"]
worldcover = auxiliary["worldcover"]
worldcover_remap = auxiliary["worldcover_remap"]
dem = auxiliary["dem"]
slope = auxiliary["slope"]
rivers_image = auxiliary["rivers_image"]

vis = us.get_vis_params()

## Availability Table

In [10]:
# Create and save the dataset availability table.

collections = {}

for event_key, event in events.items():
    label = event["name"]

    collections.update({
        f"S1 {label} pre-event ALL": s1[event_key]["pre"],
        f"S1 {label} post-event ALL": s1[event_key]["post"],
        f"S1 {label} recovery ALL": s1[event_key]["recovery"],
        f"S1 {label} pre-event ASC": s1_orbits[event_key]["pre_ASC"],
        f"S1 {label} post-event ASC": s1_orbits[event_key]["post_ASC"],
        f"S1 {label} pre-event DESC": s1_orbits[event_key]["pre_DESC"],
        f"S1 {label} post-event DESC": s1_orbits[event_key]["post_DESC"],
        f"S2 {label} pre-event raw": s2[event_key]["pre"],
        f"S2 {label} post-event raw": s2[event_key]["post"],
        f"S2 {label} recovery raw": s2[event_key]["recovery"],
    })

if INCLUDE_SEASONAL_REFERENCE:
    collections.update({
        "S1 Romagna seasonal reference ALL": s1_seasonal_ref["ALL"],
        "S1 Romagna seasonal reference ASC": s1_seasonal_ref["ASC"],
        "S1 Romagna seasonal reference DESC": s1_seasonal_ref["DESC"],
        "S2 Romagna seasonal reference raw": s2_seasonal_ref_romagna,
    })

df_availability = us.collection_sizes(
    collections,
    output_path="outputs/tables/00_dataset_availability.csv"
)

display(df_availability)

,Dataset window,Number of images
0,S1 September / Romagna pre-event ALL,14
1,S1 September / Romagna post-event ALL,3
2,S1 September / Romagna recovery ALL,10
3,S1 September / Romagna pre-event ASC,10
4,S1 September / Romagna post-event ASC,2
5,S1 September / Romagna pre-event DESC,4
6,S1 September / Romagna post-event DESC,1
7,S2 September / Romagna pre-event raw,52
8,S2 September / Romagna post-event raw,16
9,S2 September / Romagna recovery raw,25


## Visualization

In [11]:
# Create Sentinel-1 VV and Sentinel-2 RGB median preview composites.

s1_vv = {
    event_key: {
        "pre": s1[event_key]["pre"].select("VV").median().clip(events[event_key]["aoi"]),
        "post": s1[event_key]["post"].select("VV").median().clip(events[event_key]["aoi"]),
    }
    for event_key in events
}

s2_rgb = {
    event_key: {
        "pre": s2[event_key]["pre"].median().clip(events[event_key]["aoi"]),
        "post": s2[event_key]["post"].median().clip(events[event_key]["aoi"]),
    }
    for event_key in events
}

if INCLUDE_SEASONAL_REFERENCE:
    s1_seasonal_ref_vv = s1_seasonal_ref["ALL"].select("VV").median().clip(aoi_romagna)
    s2_seasonal_ref_rgb = s2_seasonal_ref_romagna.median().clip(aoi_romagna)
    s1_pre_minus_seasonal = (
        s1_vv["sept"]["pre"]
        .subtract(s1_seasonal_ref_vv)
        .rename("S1_2024_pre_minus_2023_seasonal")
    )

In [12]:
# Prepare optional seasonal reference layers for the event maps.

seasonal_layers = None

if INCLUDE_SEASONAL_REFERENCE:
    seasonal_layers = {
        "s1_vv": s1_seasonal_ref_vv,
        "s1_difference": s1_pre_minus_seasonal,
        "s2_rgb": s2_seasonal_ref_rgb,
    }

## Map 1: Romagna, September

In [13]:
# Display the September Romagna setup preview map.

Map_sept = us.make_setup_event_map(
    event_key="sept",
    events=events,
    aoi_style=event_aoi_styles["sept"],
    places_style=places_styles["sept"],
    rivers_image=rivers_image,
    s1_vv=s1_vv,
    s2_rgb=s2_rgb,
    water_occurrence=water_occurrence,
    permanent_water=permanent_water,
    vis=vis,
    include_seasonal_reference=INCLUDE_SEASONAL_REFERENCE,
    show_rivers_by_default=SHOW_RIVERS_BY_DEFAULT,
    show_s2_previews_by_default=SHOW_S2_PREVIEWS_BY_DEFAULT,
    show_water_on_event_map=SHOW_WATER_ON_EVENT_MAP,
    seasonal_layers=seasonal_layers,
)

Map_sept

Map(center=[44.35, 11.9], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…

## Map 2: Bologna, October

In [14]:
# Display the October Bologna-Pianoro setup preview map.

Map_oct = us.make_setup_event_map(
    event_key="oct",
    events=events,
    aoi_style=event_aoi_styles["oct"],
    places_style=places_styles["oct"],
    rivers_image=rivers_image,
    s1_vv=s1_vv,
    s2_rgb=s2_rgb,
    water_occurrence=water_occurrence,
    permanent_water=permanent_water,
    vis=vis,
    include_seasonal_reference=INCLUDE_SEASONAL_REFERENCE,
    show_rivers_by_default=SHOW_RIVERS_BY_DEFAULT,
    show_s2_previews_by_default=SHOW_S2_PREVIEWS_BY_DEFAULT,
    show_water_on_event_map=SHOW_WATER_ON_EVENT_MAP,
    seasonal_layers=seasonal_layers,
)

Map_oct

Map(center=[44.43, 11.35], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

## Map 3: Auxiliary

In [15]:
# Display the auxiliary datasets map.

Map_aux = us.make_auxiliary_map(
    center=[44.35, 11.8],
    zoom=8,
    aoi=aoi_emilia_romagna,
    aoi_styles=aoi_styles,
    places_style=places_styles["emilia"],
    rivers_image=rivers_image,
    auxiliary=auxiliary,
    vis=vis,
    default_layer=AUX_DEFAULT_LAYER.lower(),
)

Map_aux

Map(center=[44.35, 11.8], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…